In [12]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from pyproj import Transformer

In [ ]:
cp_path = '../../dataset/region prediction/ChinaCP_2021.tif'
qa_path = '../../dataset/region prediction/ChinaCP-DA2021.tif'

src = rasterio.open(cp_path)
crop = src.read(1)
transform = src.transform
crs = src.crs

qa = rasterio.open(qa_path).read(1)

print("Crop codes:", np.unique(crop))
print("QA codes:", np.unique(qa))

Crop codes: [-2147483648           0           3          14          15          16
          17          27         245         246         255         256]
QA codes: [         1          2          3 2147483647]


In [14]:
qa_aligned = np.empty_like(crop, dtype=qa.dtype)

reproject(
    source=rasterio.open(qa_path).read(1),
    destination=qa_aligned,
    src_transform=rasterio.open(qa_path).transform,
    src_crs=rasterio.open(qa_path).crs,
    dst_transform=transform,
    dst_crs=crs,
    resampling=Resampling.nearest
)

valid_mask = qa_aligned <= 2

print("Valid pixels:", valid_mask.sum())

Valid pixels: 34585575


In [15]:
maize_mask = np.isin(crop, [14, 245, 246]) & valid_mask
wheat_mask = np.isin(crop, [16, 246, 256]) & valid_mask

print("Maize pixels:", maize_mask.sum())
print("Wheat pixels:", wheat_mask.sum())

Maize pixels: 1746743
Wheat pixels: 732441


In [17]:
print(src.crs)

EPSG:32648


In [ ]:
transformer = Transformer.from_crs("EPSG:32648", "EPSG:4326", always_xy=True)

rows, cols = crop.shape
row_idx, col_idx = np.where(maize_mask | wheat_mask)

x, y = rasterio.transform.xy(transform, row_idx, col_idx)

lon, lat = transformer.transform(x, y)

In [19]:
df_all = pd.DataFrame({
    "lon": lon,
    "lat": lat,
    "is_maize": maize_mask[row_idx, col_idx],
    "is_wheat": wheat_mask[row_idx, col_idx]
})


In [20]:
df_maize = df_all[df_all["is_maize"] == True].copy()
df_wheat = df_all[df_all["is_wheat"] == True].copy()

df_maize["crop_type"] = "maize"
df_wheat["crop_type"] = "wheat"

In [ ]:
df_maize[["lon", "lat", "crop_type"]].to_csv(
    "../../dataset/world prediction/maize_points.csv",
    index=False
)

df_wheat[["lon", "lat", "crop_type"]].to_csv(
    "../../dataset/world prediction/wheat_points.csv",
    index=False
)

print("Saved:")
print("✔ maize_points.csv")
print("✔ wheat_points.csv")

Saved:
✔ maize_points.csv
✔ wheat_points.csv
